In [2]:
import sys
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


# Projet NoSQL & Big Data - Films TMDB (Cassandra + Redis)
## 02. Modélisation Cassandra et ingestion des données + CRUD

Ce notebook reprend là où `01_exploration_nettoyage.ipynb` s'était arrêté. Je recharge et je refais rapidement les étapes de nettoyage (pour être sûre de repartir de données propres), puis je crée les tables Cassandra et j'y insère les données, avant de faire pareil pour Redis.

## Rappel de la conception (voir plus de détail dans le README/rapport)

Je liste les questions-métier qui pilotent mes tables Cassandra (logique *query-first* : une table par façon d'interroger les données, quitte à dupliquer) :

| # | Question-métier | Table Cassandra |
|---|---|---|
| 1 | Fiche complète d'un film (id) | `movies_by_id` |
| 2 | Films les mieux notés d'un genre | `movies_by_genre` |
| 3 | Films d'un réalisateur | `movies_by_director` |
| 4 | Films dans lesquels a joué un acteur | `movies_by_actor` |
| 5 | Films d'une année, triés par popularité | `movies_by_year` |

Redis vient en complément pour ce qui doit être lu très vite : un cache par film (`movie:{id}`) et des classements (`leaderboard:top_rated`, `leaderboard:genre:{genre}`).


In [3]:
import os
import pandas as pd
import ast
from dotenv import load_dotenv

load_dotenv()
pd.set_option("display.max_columns", None)


## 1. Recharger et renettoyer les données

Je refais les étapes du notebook précédent en condensé, pour que ce notebook soit autonome (je n'ai pas besoin d'avoir exécuté l'autre avant).


In [4]:
def parse_names(raw_value):
    try:
        return [item["name"] for item in ast.literal_eval(raw_value)]
    except Exception:
        return []

def get_director(raw_crew):
    try:
        for person in ast.literal_eval(raw_crew):
            if person.get("job") == "Director":
                return person.get("name")
    except Exception:
        pass
    return None

def get_main_cast(raw_cast, n=5):
    return parse_names(raw_cast)[:n]

movies = pd.read_csv("../tmdb_5000_movies.csv")
credits = pd.read_csv("../tmdb_5000_credits.csv")

movies["genres_list"] = movies["genres"].apply(parse_names)
credits["director"] = credits["crew"].apply(get_director)
credits["main_cast"] = credits["cast"].apply(get_main_cast)

df = movies.merge(credits[["movie_id", "director", "main_cast"]], left_on="id", right_on="movie_id", how="left")
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["budget_clean"] = df["budget"].replace(0, pd.NA)
df["revenue_clean"] = df["revenue"].replace(0, pd.NA)

# released : sous-ensemble utilisé pour calculer la note pondérée (a besoin de vote_average/vote_count fiables)
released = df[df["status"] == "Released"].copy()

C = released["vote_average"].mean()
m = released["vote_count"].quantile(0.80)
released["weighted_rating"] = released.apply(
    lambda row: (row["vote_count"] / (row["vote_count"] + m) * row["vote_average"]) + (m / (row["vote_count"] + m) * C),
    axis=1
)

# on ramène weighted_rating dans le dataframe complet : NaN pour les films non sortis (normal, pas de note fiable)
df = df.merge(released[["id", "weighted_rating"]], on="id", how="left")

print(f"{len(df)} films au total, {df['weighted_rating'].notna().sum()} avec une note pondérée calculée.")


4803 films au total, 4795 avec une note pondérée calculée.


## 2. Connexion aux bases

Même code que dans `test_connexion.py`, cette fois je garde les objets `session` et `redis_client` pour la suite du notebook.


In [5]:
from cassandra.cluster import Cluster, ProtocolVersion
from cassandra.auth import PlainTextAuthProvider

cloud_config = {"secure_connect_bundle": os.environ["ASTRA_DB_BUNDLE_PATH"], "connect_timeout": 30}
auth_provider = PlainTextAuthProvider(username="token", password=os.environ["ASTRA_DB_APPLICATION_TOKEN"])
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider, protocol_version=ProtocolVersion.V4)
session = cluster.connect()
print("Connectée à Cassandra.")


Connectée à Cassandra.


In [6]:
import redis

redis_client = redis.Redis(
    host=os.environ["REDIS_HOST"],
    port=int(os.environ["REDIS_PORT"]),
    password=os.environ["REDIS_PASSWORD"],
    username=os.environ.get("REDIS_USERNAME", "default"),
    decode_responses=True,
)
print("Connectée à Redis :", redis_client.ping())


Connectée à Redis : True


## 3. Création du keyspace et des tables

Une seule fois nécessaire - si je relance cette cellule plus tard, `IF NOT EXISTS` évite une erreur si les tables existent déjà.


In [7]:
session.set_keyspace("tmdb_platform")

session.execute("""
CREATE TABLE IF NOT EXISTS movies_by_id (
    movie_id int PRIMARY KEY,
    title text,
    director text,
    main_cast list<text>,
    genres list<text>,
    release_year int,
    release_date text,
    runtime float,
    budget bigint,
    revenue bigint,
    popularity float,
    vote_average float,
    vote_count int,
    weighted_rating float,
    original_language text,
    status text
);
""")

session.execute("""
CREATE TABLE IF NOT EXISTS movies_by_genre (
    genre text,
    weighted_rating float,
    movie_id int,
    title text,
    vote_count int,
    PRIMARY KEY (genre, weighted_rating, movie_id)
) WITH CLUSTERING ORDER BY (weighted_rating DESC);
""")

session.execute("""
CREATE TABLE IF NOT EXISTS movies_by_director (
    director text,
    release_year int,
    movie_id int,
    title text,
    weighted_rating float,
    PRIMARY KEY (director, release_year, movie_id)
) WITH CLUSTERING ORDER BY (release_year DESC);
""")

session.execute("""
CREATE TABLE IF NOT EXISTS movies_by_actor (
    actor_name text,
    movie_id int,
    title text,
    release_year int,
    PRIMARY KEY (actor_name, movie_id)
);
""")

session.execute("""
CREATE TABLE IF NOT EXISTS movies_by_year (
    release_year int,
    popularity float,
    movie_id int,
    title text,
    PRIMARY KEY (release_year, popularity, movie_id)
) WITH CLUSTERING ORDER BY (popularity DESC);
""")

print("Keyspace sélectionné et 5 tables créées (ou déjà existantes).")

Keyspace sélectionné et 5 tables créées (ou déjà existantes).


## 4. Insertion des données dans Cassandra

J'utilise des requêtes préparées (`session.prepare`) - le driver compile la requête une seule fois plutôt qu'à chaque ligne, et ça évite les soucis d'échappement avec les titres contenant des apostrophes (ex: "Ocean's Eleven").

Sur ~4700 films, cette cellule peut prendre 5 à 8 minutes selon la latence réseau vers Astra.

In [10]:
from cassandra.concurrent import execute_concurrent_with_args
import time

session.set_keyspace("tmdb_platform")

insert_by_id = session.prepare("""
    INSERT INTO movies_by_id
    (movie_id, title, director, main_cast, genres, release_year, release_date,
     runtime, budget, revenue, popularity, vote_average, vote_count,
     weighted_rating, original_language, status)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")
insert_by_genre = session.prepare("INSERT INTO movies_by_genre (genre, weighted_rating, movie_id, title, vote_count) VALUES (?, ?, ?, ?, ?)")
insert_by_director = session.prepare("INSERT INTO movies_by_director (director, release_year, movie_id, title, weighted_rating) VALUES (?, ?, ?, ?, ?)")
insert_by_actor = session.prepare("INSERT INTO movies_by_actor (actor_name, movie_id, title, release_year) VALUES (?, ?, ?, ?)")
insert_by_year = session.prepare("INSERT INTO movies_by_year (release_year, popularity, movie_id, title) VALUES (?, ?, ?, ?)")

CONCURRENCY = 100  # nombre de requêtes envoyées en parallèle - 100 est une valeur raisonnable pour un tier gratuit

def run_concurrent(label, stmt, params_list):
    """Envoie toutes les requêtes de params_list en parallèle (au lieu d'une par une),
    puis vérifie s'il y a eu des erreurs."""
    t0 = time.time()
    results = execute_concurrent_with_args(session, stmt, params_list, concurrency=CONCURRENCY)
    errors = [(i, r) for i, (success, r) in enumerate(results) if not success]
    elapsed = time.time() - t0
    print(f"{label} : {len(params_list)} lignes en {elapsed:.1f}s ({len(errors)} erreurs)")
    if errors[:3]:
        print("   exemples d'erreurs :", errors[:3])
    return errors

# --- 1. Je prépare TOUTES les listes de paramètres d'abord (rapide, pas de réseau ici) ---

params_by_id = []
params_by_actor = []
for _, row in df.iterrows():
    budget_val = None if pd.isna(row["budget_clean"]) else int(row["budget_clean"])
    revenue_val = None if pd.isna(row["revenue_clean"]) else int(row["revenue_clean"])
    runtime_val = float(row["runtime"]) if pd.notna(row["runtime"]) else None
    weighted_rating_val = float(row["weighted_rating"]) if pd.notna(row["weighted_rating"]) else None
    release_year_val = int(row["release_year"]) if pd.notna(row["release_year"]) else None
    release_date_val = None if pd.isna(row["release_date"]) else str(row["release_date"])
    director_val = None if pd.isna(row["director"]) else row["director"]

    params_by_id.append((
        int(row["id"]), row["title"], director_val, row["main_cast"], row["genres_list"],
        release_year_val, release_date_val, runtime_val,
        budget_val, revenue_val, float(row["popularity"]), float(row["vote_average"]),
        int(row["vote_count"]), weighted_rating_val, row["original_language"], row["status"],
    ))
    for actor in row["main_cast"]:
        params_by_actor.append((actor, int(row["id"]), row["title"], release_year_val))

params_by_genre = []
for _, row in released.iterrows():
    for genre in row["genres_list"]:
        params_by_genre.append((genre, float(row["weighted_rating"]), int(row["id"]), row["title"], int(row["vote_count"])))

params_by_year = []
for _, row in released.dropna(subset=["release_year"]).iterrows():
    params_by_year.append((int(row["release_year"]), float(row["popularity"]), int(row["id"]), row["title"]))

params_by_director = []
skipped_no_director = 0
for _, row in released.dropna(subset=["release_year"]).iterrows():
    if pd.isna(row["director"]):
        skipped_no_director += 1
        continue
    params_by_director.append((row["director"], int(row["release_year"]), int(row["id"]), row["title"], float(row["weighted_rating"])))

# --- 2. J'envoie chaque lot en parallèle ---

run_concurrent("movies_by_id", insert_by_id, params_by_id)
run_concurrent("movies_by_actor", insert_by_actor, params_by_actor)
run_concurrent("movies_by_genre", insert_by_genre, params_by_genre)
run_concurrent("movies_by_year", insert_by_year, params_by_year)
run_concurrent("movies_by_director", insert_by_director, params_by_director)

print(f"\nTerminé. {skipped_no_director} films sautés dans movies_by_director faute de réalisateur identifié.")

movies_by_id : 4803 lignes en 6.6s (0 erreurs)
movies_by_actor : 23594 lignes en 45.2s (0 erreurs)
movies_by_genre : 12144 lignes en 18.4s (0 erreurs)
movies_by_year : 4794 lignes en 8.4s (0 erreurs)
movies_by_director : 4767 lignes en 9.6s (0 erreurs)

Terminé. 27 films sautés dans movies_by_director faute de réalisateur identifié.


In [11]:
for table in ["movies_by_id", "movies_by_actor", "movies_by_genre", "movies_by_year", "movies_by_director"]:
    count = session.execute(f"SELECT COUNT(*) FROM {table}").one()
    print(f"{table} : {count.count} lignes")

movies_by_id : 4803 lignes
movies_by_actor : 23589 lignes
movies_by_genre : 12144 lignes
movies_by_year : 4794 lignes
movies_by_director : 4767 lignes


### Vérification

Je relis quelques lignes pour confirmer que l'insertion a fonctionné, avant de passer à Redis.


In [12]:
rows = session.execute("SELECT title, director, weighted_rating FROM movies_by_id LIMIT 5")
for r in rows:
    print(r)

count = session.execute("SELECT COUNT(*) FROM movies_by_id").one()
print()
print("Nombre total de films dans movies_by_id :", count.count)


Row(title='School of Rock', director='Richard Linklater', weighted_rating=6.454816818237305)
Row(title='She Wore a Yellow Ribbon', director='John Ford', weighted_rating=6.153509616851807)
Row(title='House Party 2', director='Doug McHenry', weighted_rating=6.063448905944824)
Row(title='Tank Girl', director='Rachel Talalay', weighted_rating=6.018669128417969)
Row(title='Need for Speed', director='Scott Waugh', weighted_rating=6.097410202026367)

Nombre total de films dans movies_by_id : 4803


## 5. Population de Redis

Cassandra reste ma source de vérité, Redis n'est qu'une couche de cache/classement construite par-dessus. Je ne classe que les films avec un minimum de votes (`VOTE_COUNT_MIN`), pour éviter qu'un film avec 2 votes de 10/10 ne fausse le classement.


In [15]:
VOTE_COUNT_MIN = 50

for _, row in released.iterrows():
    movie_key = f"movie:{int(row['id'])}"
    redis_client.hset(movie_key, mapping={
        "title": row["title"],
        "director": row["director"] if pd.notna(row["director"]) else "",
        "genres": "|".join(row["genres_list"]),
        "release_year": int(row["release_year"]) if pd.notna(row["release_year"]) else "",
        "vote_average": float(row["vote_average"]),
        "weighted_rating": float(row["weighted_rating"]),
    })
    if row["vote_count"] >= VOTE_COUNT_MIN:
        redis_client.zadd("leaderboard:top_rated", {str(int(row["id"])): float(row["weighted_rating"])})
        for genre in row["genres_list"]:
            redis_client.zadd(f"leaderboard:genre:{genre}", {str(int(row["id"])): float(row["weighted_rating"])})

redis_client.set("stats:total_movies", len(released))
print("Redis alimenté.")


Redis alimenté.


In [16]:
suspects = redis_client.keys("movie:*")
count_nan = 0
for key in suspects:
    if redis_client.hget(key, "director") == "nan":
        count_nan += 1
print(f"{count_nan} fiches Redis avec 'nan' au lieu d'une valeur vide.")

0 fiches Redis avec 'nan' au lieu d'une valeur vide.


In [14]:
top5 = redis_client.zrevrange("leaderboard:top_rated", 0, 4, withscores=True)
for movie_id, score in top5:
    infos = redis_client.hgetall(f"movie:{movie_id}")
    print(f"{infos['title']} ({infos['release_year']}) - {score:.2f}")


The Shawshank Redemption (1994) - 8.25
Fight Club (1999) - 8.10
The Godfather (1972) - 8.08
Pulp Fiction (1994) - 8.07
The Dark Knight (2008) - 8.04


## Bilan

- Keyspace `tmdb_platform` et 5 tables créées sur Astra DB.
- `movies_by_id` et `movies_by_actor` : catalogue complet, les 4803 films (Rumored/Post Production compris, `weighted_rating` vide pour ces 8 films puisqu'ils n'ont pas de note fiable).
- `movies_by_genre`, `movies_by_year`, `movies_by_director` : filtrées chacune selon ce qu'exige réellement sa propre clé primaire (films sortis, + `release_year` non vide pour les deux dernières, + `director` non vide en plus pour `movies_by_director` - les films sans réalisateur identifié sont comptés et affichés, pas juste perdus silencieusement).
- Redis alimenté en cache par film + classements (global et par genre), en filtrant les films avec trop peu de votes.

Prochaine section : le rapport analytique avec les vraies requêtes.


## 6. CRUD : Create, Read, Update, Delete

L'énoncé demande de démontrer les 4 opérations de base en Python, sur les deux bases. Je travaille sur un film de test avec un movie_id qui n'existe pas dans le vrai dataset (999999), pour ne jamais risquer d'écraser une vraie donnée pendant la démo.


**a. CREATE (Cassandra)**

In [17]:
TEST_MOVIE_ID = 999999
 
session.execute(insert_by_id, (
    TEST_MOVIE_ID, "Mon Film Test", "Réalisateur Test",
    ["Acteur A", "Acteur B"], ["Comedy", "Drama"],
    2024, "2024-01-01", 100.0,
    1_000_000, 2_000_000, 5.0, 7.0, 42, 6.5, "fr", "Released",
))
 
print("Film test créé dans movies_by_id.")

Film test créé dans movies_by_id.


**b. READ (Cassandra)**

In [18]:
rows = session.execute(
    "SELECT movie_id, title, director, vote_average FROM movies_by_id WHERE movie_id = %s",
    (TEST_MOVIE_ID,)
)
for r in rows:
    print(r)

Row(movie_id=999999, title='Mon Film Test', director='Réalisateur Test', vote_average=7.0)


**c. UPDATE (Cassandra) : un piège à connaître**

Contrairement à SQL, `UPDATE ... WHERE colonne_quelconque = valeur` n'existe pas en Cassandra. Je ne peux mettre à jour qu'en fournissant la clé primaire complète — ici `movie_id`, puisque c'est la seule clé de `movies_by_id`. Impossible par exemple de faire `UPDATE movies_by_id SET ... WHERE title = ...`.


In [19]:
session.execute(
    "UPDATE movies_by_id SET vote_average = %s WHERE movie_id = %s",
    (8.0, TEST_MOVIE_ID)
)
 
rows = session.execute(
    "SELECT title, vote_average FROM movies_by_id WHERE movie_id = %s",
    (TEST_MOVIE_ID,)
)
for r in rows:
    print("Après update :", r)

Après update : Row(title='Mon Film Test', vote_average=8.0)


**d. DELETE (Cassandra)**

In [20]:
session.execute("DELETE FROM movies_by_id WHERE movie_id = %s", (TEST_MOVIE_ID,))
 
check = list(session.execute(
    "SELECT * FROM movies_by_id WHERE movie_id = %s", (TEST_MOVIE_ID,)
))
print("Film supprimé, résultat de la relecture :", check)  # doit afficher []

Film supprimé, résultat de la relecture : []


### CRUD côté Redis

Redis fonctionne différemment : `HSET` sert à la fois de Create et d'Update (s'il n'existe pas encore, il le crée ; s'il existe déjà, il écrase les champs donnés) — pas de distinction entre les deux comme en SQL ou en CQL.


In [21]:
redis_test_key = f"movie:{TEST_MOVIE_ID}"
 
# Create (première fois que la clé existe)
redis_client.hset(redis_test_key, mapping={
    "title": "Mon Film Test",
    "director": "Réalisateur Test",
    "vote_average": 7.0,
})
print("Create :", redis_client.hgetall(redis_test_key))
 
# Update (même commande HSET, sur un champ déjà existant)
redis_client.hset(redis_test_key, "vote_average", 8.0)
print("Update :", redis_client.hgetall(redis_test_key))
 
# Read
print("Read   :", redis_client.hgetall(redis_test_key))
 
# Delete
redis_client.delete(redis_test_key)
print("Existe encore après delete :", redis_client.exists(redis_test_key))

Create : {'title': 'Mon Film Test', 'director': 'Réalisateur Test', 'vote_average': '7.0'}
Update : {'title': 'Mon Film Test', 'director': 'Réalisateur Test', 'vote_average': '8.0'}
Read   : {'title': 'Mon Film Test', 'director': 'Réalisateur Test', 'vote_average': '8.0'}
Existe encore après delete : 0


### Limite du modèle dénormalisé : la cohérence entre tables

Ce CRUD ne touche que `movies_by_id` (et sa clé Redis équivalente). En réalité, si je modifiais le titre d'un vrai film en production, il faudrait répercuter le changement à la main dans les 4 autres tables Cassandra (`movies_by_genre`, `movies_by_director`, `movies_by_actor`, `movies_by_year`), puisqu'elles dupliquent volontairement les données pour chaque façon d'interroger le catalogue — contrairement à SQL, il n'y a pas de contrainte d'intégrité référentielle qui ferait ça automatiquement. C'est le compromis assumé du query-first design : des lectures plus rapides, mais la cohérence entre tables devient une responsabilité applicative (à gérer via un script de synchronisation, ou en acceptant que seule `movies_by_id` soit modifiable et que les autres tables soient reconstruites périodiquement).
